In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors

from skimage.measure import label
import pandas as pd
from tqdm.auto import tqdm

# Bioflore Dataset Size

In [ ]:
gdf_path = "/home/luizluz/Documentos/multi-task-fcn/matematica_industria_data/raw/labels/labels.shp"
gdf = gpd.read_file(gdf_path)

In [ ]:
from glob import glob
import rasterio
tiff_paths = glob("../../matematica_industria_data/raw/geotiffs/*.tif")

datasets = [rasterio.open(tiff_path) for tiff_path in tiff_paths]

In [ ]:
from os.path import abspath
print(
    '"'+'" "'.join([abspath(tiff_path) for tiff_path in tiff_paths])+'"'
)


In [ ]:
import numpy as np
from shapely.geometry import box

# Get the CRS from the first raster (assuming all rasters share the same CRS)
raster_crs = datasets[0].crs

# Reproject gdf to match the raster CRS if they differ
if gdf.crs != raster_crs:
    gdf = gdf.to_crs(raster_crs)
    # print(f"Reprojected GeoDataFrame from {gdf.crs} to {raster_crs}")

# Create a list of raster bounding boxes and their corresponding filenames
raster_infos = []
for d in datasets:
    bounds = d.bounds
    raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    raster_infos.append({'name': d.name, 'bbox': raster_bbox})

def find_raster_for_shape(shape):
    geom = shape.geometry
    for raster_info in raster_infos:
        if raster_info['bbox'].intersects(geom):
            return raster_info['name']
    return None

gdf['tiff_file'] = gdf.apply(find_raster_for_shape, axis=1)

# É possível trabalhar com apenas uma subamostra de todas as imagens que temos?

## Temos geotiffs que possuem exatamente as mesmas espécies?

In [ ]:
from itertools import combinations

# Cria um dicionário: geotiff -> set de espécies
geotiff_species = gdf.groupby("geotiff")["species"].apply(set).to_dict()

# Lista para registrar pares que possuem exatamente as mesmas espécies
geotiff_pairs_with_exact_species = []

# Testa todos os pares de geotiffs
for g1, g2 in combinations(geotiff_species.keys(), 2):
    if geotiff_species[g1] == geotiff_species[g2]:
        geotiff_pairs_with_exact_species.append((g1, g2, geotiff_species[g1]))

if geotiff_pairs_with_exact_species:
    print("Existem pares de geotiffs com exatamente as mesmas espécies:")
    for g1, g2, species_set in geotiff_pairs_with_exact_species:
        print(f"{g1} e {g2} possuem exatamente estas espécies: {sorted(list(species_set))}")
else:
    print("Não existem geotiffs diferentes contendo exatamente as mesmas espécies.")


## Temos um conjunto de geotiffs que possuem a mesma subamostra?


In [ ]:
gdf.groupby("species")["geotiff"].apply(lambda x: x.nunique()).sort_values(ascending=False)

In [ ]:
# Definindo o conjunto das espécies de interesse
target_species = {
    'Tachigali aurea', 
    'Pterodon emarginatus', 
    'Qualea parviflora', 
    'Salvertia convallariodora'
}

# Cria um dicionário: geotiff -> set de espécies (caso não tenha sido criado acima)
geotiff_species = gdf.groupby("geotiff")["species"].apply(set)

# Seleciona os geotiffs que possuem todas as espécies desejadas
geotiffs_with_all_target_species = geotiff_species[geotiff_species.apply(lambda s: target_species.issubset(s))]

print("Geotiffs que possuem TODAS as espécies alvo:")
for geotiff in geotiffs_with_all_target_species.index:
    print(geotiff)

mosaics = geotiffs_with_all_target_species.index

# Quantidade de amostras em cada mosaico específico
counts = [gdf[gdf["geotiff"].str.contains(mosaic) & gdf["species"].isin(target_species)].shape[0] for mosaic in mosaics]
# Quantidade total de amostras
total_samples = sum(counts)

for i, mosaic in enumerate(mosaics):
    print(f"Quantidade de amostras em {mosaic}: {counts[i]}")
    
print(f"Quantidade total de amostras: {total_samples}")

# Quantidade de amostras por espécie
species_counts = gdf[gdf["species"].isin(target_species) & gdf["geotiff"].isin(mosaics)]["species"].value_counts()
species_counts.head(10)

# Seleciona os shapesfiles que serão usados

In [ ]:
import os
OUT_DIR = "/home/luizluz/Documentos/multi-task-fcn/bioflore_data"

In [ ]:
gdf_selected = gdf[gdf["geotiff"].isin(mosaics) & gdf["species"].isin(target_species)].copy()

gdf_selected["old_label"] = gdf["label"].copy()
# Map species to new labels starting from 1 to n (where n is the number of target species)
unique_species = sorted(gdf_selected["species"].unique())
species_to_new_label = {sp: i+1 for i, sp in enumerate(unique_species)}
gdf_selected["label"] = gdf_selected["species"].map(species_to_new_label)

In [ ]:
gdf_selected[["label", "old_label"]].value_counts()

In [ ]:

os.makedirs(os.path.join(OUT_DIR, "shapes"), exist_ok=True)
gdf_selected.to_file(os.path.join(OUT_DIR, "shapes", "labels.shp"))

In [ ]:
from sklearn.model_selection import train_test_split

# Split into 80% train and 20% test
gdf_train, gdf_test = train_test_split(gdf_selected, test_size=0.2, random_state=42, stratify=gdf_selected["species"])

print("Train samples:", len(gdf_train))
print("Test samples:", len(gdf_test))


In [ ]:
gdf_train["species"].value_counts()

In [ ]:
gdf_test["species"].value_counts()

In [ ]:
gdf_train.to_file(os.path.join(OUT_DIR, "shapes", "train_labels.shp"))
gdf_test.to_file(os.path.join(OUT_DIR, "shapes", "test_labels.shp"))


In [ ]:

gdf_train["label"].drop_duplicates().sort_values()

# Save as raster

In [ ]:
OUTPUT_RASTER_DIR= "/home/luizluz/Documentos/multi-task-fcn/bioflore_data/input_data/segmentations"

In [ ]:
# Verificar quais colunas temos disponíveis
print("Colunas do gdf_train:")
print(gdf_train.columns.tolist())
print("\nValores únicos da coluna 'label':")
print(gdf_train["label"].drop_duplicates().sort_values().tolist())


In [ ]:
# Rasterizar usando a coluna 'label' para definir os valores dos pixels
import os
from rasterio.features import rasterize

# Create output directories
os.makedirs(os.path.join(OUTPUT_RASTER_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_RASTER_DIR, "test"), exist_ok=True)

# Map filenames to dataset objects for easy lookup
dataset_map = {os.path.basename(d.name): d for d in datasets}

for mosaic_name in mosaics:
    # Use basename just in case
    key = os.path.basename(mosaic_name)
    
    if key not in dataset_map:
        print(f"Warning: Raster for {mosaic_name} not found in open datasets.")
        continue
        
    src = dataset_map[key]
    out_meta = src.meta.copy()
    # Ensure single band, uint8, nodata=0
    out_meta.update({"count": 1, "dtype": rasterio.uint8, "nodata": 0})
    
    # --- Process Train Data ---
    train_shapes = gdf_train[gdf_train["geotiff"] == mosaic_name]
    
    train_mask = np.zeros(src.shape, dtype=rasterio.uint8)
    if not train_shapes.empty:
        shapes_to_rasterize = []
        for _, row in train_shapes.iterrows():
            if row.geometry:
                # Usar a coluna 'label' para definir o valor do pixel
                val = int(row["label"])
                shapes_to_rasterize.append((row.geometry, val))
        
        if shapes_to_rasterize:
            try:
                train_mask = rasterize(
                    shapes=shapes_to_rasterize,
                    out_shape=src.shape,
                    transform=src.transform,
                    fill=0,
                    dtype=rasterio.uint8
                )
            except Exception as e:
                print(f"Error rasterizing train for {mosaic_name}: {e}")

    # Save Train Raster
    train_out_path = os.path.join(OUTPUT_RASTER_DIR, "train", key)
    with rasterio.open(train_out_path, "w", **out_meta) as dest:
        dest.write(train_mask, 1)
        
    # --- Process Test Data ---
    test_shapes = gdf_test[gdf_test["geotiff"] == mosaic_name]
    
    test_mask = np.zeros(src.shape, dtype=rasterio.uint8)
    if not test_shapes.empty:
        shapes_to_rasterize = []
        for _, row in test_shapes.iterrows():
            if row.geometry:
                # Usar a coluna 'label' para definir o valor do pixel
                val = int(row["label"])
                shapes_to_rasterize.append((row.geometry, val))
        
        if shapes_to_rasterize:
            try:
                test_mask = rasterize(
                    shapes=shapes_to_rasterize,
                    out_shape=src.shape,
                    transform=src.transform,
                    fill=0,
                    dtype=rasterio.uint8
                )
            except Exception as e:
                print(f"Error rasterizing test for {mosaic_name}: {e}")

    # Save Test Raster
    test_out_path = os.path.join(OUTPUT_RASTER_DIR, "test", key)
    with rasterio.open(test_out_path, "w", **out_meta) as dest:
        dest.write(test_mask, 1)

    print(f"Saved segmentation masks for {mosaic_name}")
